#  Automation - Notebook Option B: Create a Task with EXECUTE NOTEBOOK

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.core import Root
from snowflake.core.task import Task, Cron

session = get_active_session()

database_name = "ML_LAB"
schema_name = "DATA"
warehouse_name = "COMPUTE_WH"

session.use_database(database_name)
session.use_schema(schema_name)

api_root = Root(session)
schema = api_root.databases[database_name].schemas[schema_name]
tasks = schema.tasks

task_name = "execute_inference_notebook"

iris_task = Task(
    task_name,
    definition="EXECUTE NOTEBOOK ML_LAB.MODELS.iris_inference_notebook()",
    warehouse=warehouse_name,
    schedule=Cron("0 7 * * *", "UTC")
)

tasks.create(iris_task, mode="or_replace")

task_ref = tasks[task_name]
task_ref.resume()

print(f"Task '{task_name}' created and resumed.")

# Optional - SQL version of the same above

In [ ]:
%%sql -r dataframe_2
CREATE OR REPLACE TASK execute_inference_notebook
  WAREHOUSE = COMPUTE_WH
  SCHEDULE = 'USING CRON 0 7 * * * UTC'
AS
  EXECUTE NOTEBOOK ML_LAB.NOTEBOOKS.inference();

ALTER TASK run_iris_inference_notebook RESUME;

-- Check run history
SELECT *
FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(TASK_NAME => 'RUN_IRIS_INFERENCE_NOTEBOOK'))
ORDER BY SCHEDULED_TIME DESC;